# USER BEHAVIOR PREDICTION — DATAFLOW 2026
## GRU MODEL

In [1]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/hungdangdinh/user-behavior-prediction-dataset/sample_submission.csv
/kaggle/input/datasets/hungdangdinh/user-behavior-prediction-dataset/X_test.csv
/kaggle/input/datasets/hungdangdinh/user-behavior-prediction-dataset/Y_val.csv
/kaggle/input/datasets/hungdangdinh/user-behavior-prediction-dataset/Y_train.csv
/kaggle/input/datasets/hungdangdinh/user-behavior-prediction-dataset/X_train.csv
/kaggle/input/datasets/hungdangdinh/user-behavior-prediction-dataset/X_val.csv


### Import library and Initial Config

Khai báo các thư viện cần thiết trong quá trình huấn luyện mô hình, ngoài ra khai báo các biến, đường dẫn thư mục, siêu tham số cho mô hình

In [2]:
import os, random, argparse, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

warnings.filterwarnings("ignore")

In [3]:
CFG = {
    # ── Dữ liệu
    "x_train":      "/kaggle/input/datasets/hungdangdinh/user-behavior-prediction-dataset/X_train.csv",
    "y_train":      "/kaggle/input/datasets/hungdangdinh/user-behavior-prediction-dataset/Y_train.csv",
    "x_val":        "/kaggle/input/datasets/hungdangdinh/user-behavior-prediction-dataset/X_val.csv",
    "y_val":        "/kaggle/input/datasets/hungdangdinh/user-behavior-prediction-dataset/Y_val.csv",
    "x_test":       "/kaggle/input/datasets/hungdangdinh/user-behavior-prediction-dataset/X_test.csv",
    "output_file":  "/kaggle/working/submission.csv",

    # ── Chuỗi
    "max_seq_len":  30,          # Đặt = P95 của train sau khi chạy EDA
    "pad_value":    0,           # Index 0 = PAD token
    "unk_value":    1,           # Index 1 = UNK token

    # ── Model
    "model_type":   "gru",    # "bilstm" | "cnn" | "transformer" | "gru"
    "embed_dim":    256,
    "hidden_dim":   512,
    "n_layers":     2,
    "n_heads":      4,           # Chỉ dùng cho Transformer
    "dropout":      0.4,
    "head_hidden":  64, # Hidden dim của mỗi output head
    "bidirectional": True,
    # ── Training
    "batch_size":   256,
    "epochs":       100,
    "lr":           1e-3,
    "weight_decay": 1e-5,
    "patience":     10,           # Early stopping patience
    "seed":         42,
    "num_workers":  0,
    "device":       "cuda" if torch.cuda.is_available() else "cpu",
    "save_path":    "/kaggle/working/model_bigru.pt",
}

FEATURE_COLS = [f"feature_{i}" for i in range(1, 38)]
TARGET_COLS  = [f"attr_{i}"    for i in range(1, 7)]

In [ ]:
def seed_everything(seed: int):
    """
    Đặt seed cho tất cả các thư viện để đảm bảo kết quả có thể tái lập.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG["seed"])

### Xây dựng Vocab và Chuẩn bị dữ liệu Dataloader cho mô hình

In [5]:
class Vocabulary:
    """
    Map token (số nguyên gốc) → index liên tục bắt đầu từ 2.
    Index 0 = PAD, Index 1 = UNK.
    """
    PAD_IDX = 0
    UNK_IDX = 1

    def __init__(self):
        self.token2idx = {}
        self.next_idx  = 2          # 0=PAD, 1=UNK đã chiếm trước

    def build(self, X: pd.DataFrame):
        """Duyệt toàn bộ X để thu thập vocab từ tập TRAIN."""
        for col in FEATURE_COLS:
            for val in X[col].dropna().unique():
                if val not in self.token2idx:
                    self.token2idx[val] = self.next_idx
                    self.next_idx += 1
        print(f"  Vocabulary size: {self.next_idx} (PAD=0, UNK=1, tokens={self.next_idx-2})")

    def encode(self, val) -> int:
        return self.token2idx.get(val, self.UNK_IDX)

    @property
    def size(self) -> int:
        return self.next_idx

In [6]:
class BehaviorDataset(Dataset):
    """
    Mỗi sample gồm:
      - seq   : LongTensor [max_seq_len] — chuỗi token đã pad (Right-PAD)
      - mask  : BoolTensor [max_seq_len] — True = token thực, False = PAD
      - meta  : FloatTensor [3]          — [seq_len_norm, unique_ratio, first_token_norm]
      - labels: LongTensor [6]           — 6 target (nếu có Y, else zeros)
    """
    def __init__(self,
                 X: pd.DataFrame,
                 vocab: Vocabulary,
                 max_seq_len: int,
                 Y: pd.DataFrame = None,
                 n_classes: list  = None):
        self.X          = X
        self.Y          = Y     
        self.vocab      = vocab
        self.max_seq_len = max_seq_len
        self.ids        = X.index.tolist()
        self.n_classes  = n_classes   # list[int] — số class mỗi attr (để encode label)

        # Tính label encoding nếu có Y
        self.label_encoders = {}
        if Y is not None and n_classes is not None:
            for i, col in enumerate(TARGET_COLS):
                uniq = sorted(Y[col].unique())
                self.label_encoders[col] = {v: j for j, v in enumerate(uniq)}

    def __len__(self):
        return len(self.ids)

    def _encode_row(self, idx):
        """Mã hoá một hàng X thành token sequence + mask + meta."""
        row   = self.X.loc[idx, FEATURE_COLS]
        tokens_raw = row.dropna().values          # Lấy token thực (bỏ NaN)
        seq_len    = len(tokens_raw)

        # Encode token → index
        encoded = [self.vocab.encode(t) for t in tokens_raw]

        # Truncate nếu quá dài (giữ phần cuối — recent context)
        if seq_len > self.max_seq_len:
            encoded  = encoded[-self.max_seq_len:]
            seq_len  = self.max_seq_len

        # RIGHT-PAD (mới)
        pad_len = self.max_seq_len - seq_len
        seq  = encoded + [self.vocab.PAD_IDX] * pad_len   # PAD ở PHẢI
        mask = [True] * seq_len + [False]     * pad_len

        # Meta-features (chuẩn hoá thô)
        unique_ratio = len(set(encoded)) / max(seq_len, 1)
        # first_token  = (encoded[0] / self.vocab.size) if seq_len > 0 else 0.0
        seq_len_norm = seq_len / self.max_seq_len

        return seq, mask, [seq_len_norm, unique_ratio]

    def __getitem__(self, i):
        idx = self.ids[i]
        seq, mask, meta = self._encode_row(idx)

        seq_t  = torch.tensor(seq,  dtype=torch.long)
        mask_t = torch.tensor(mask, dtype=torch.bool)
        meta_t = torch.tensor(meta, dtype=torch.float)

        if self.Y is not None:
            labels = []
            for col in TARGET_COLS:
                raw_label = self.Y.loc[idx, col]
                if col in self.label_encoders:
                    label = self.label_encoders[col].get(raw_label, 0)
                else:
                    label = int(raw_label)
                labels.append(label)
            label_t = torch.tensor(labels, dtype=torch.long)
            return seq_t, mask_t, meta_t, label_t

        # Tập Test không có Y
        return seq_t, mask_t, meta_t

### Khai báo các Class liên quan đến kiến trúc của mô hình

In [ ]:
class ClassificationHead(nn.Module):
    """
    MLP 2 lớp cho mỗi attr: [context_dim + meta_dim] → n_classes
    Sử dụng sau khi đã gộp context vector và meta-features.
    """
    def __init__(self, in_dim: int, hidden_dim: int, n_classes: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class AttentionPooling(nn.Module):
    """
    Class AttentionPooling để gộp thông tin từ các bước thời gian thành một vector ngữ cảnh duy nhất.
    Sử dụng attention mechanism để học trọng số cho mỗi bước thời gian.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x, mask=None):
        attn_logits = self.attn(x) # [B, L, 1]
        
        if mask is not None:
            # Mask có dạng [B, L], cần biến đổi để cộng vào logits
            # Những vị trí mask=0 (padding) sẽ bị trừ đi một số rất lớn để softmax ra 0
            mask = mask.unsqueeze(-1) # [B, L, 1]
            attn_logits = attn_logits.masked_fill(mask == 0, -1e9)
            
        attn_weights = torch.softmax(attn_logits, dim=1)
        context_vector = torch.sum(x * attn_weights, dim=1)
        return context_vector 

In [ ]:
class GRUEncoder(nn.Module):
    """Mã hoá chuỗi bằng GRU 2 chiều, sau đó gộp thông tin qua AttentionPooling."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(
            embed_dim, 
            hidden_dim, 
            n_layers, 
            batch_first=True, 
            bidirectional=True, 
            dropout=dropout if n_layers > 1 else 0
        )
        self.out_dim = hidden_dim * 2 
        self.attention = AttentionPooling(self.out_dim) 
        
    def forward(self, x, mask=None):
        # x: [batch_size, seq_len]
        # mask: [batch_size, seq_len] - chưa dùng đến trong GRU cơ bản nhưng cần nhận vào để tránh lỗi TypeError
        
        x = self.embedding(x) # [batch, seq_len, embed_dim]
        
        # Chạy qua GRU
        out, _ = self.gru(x) # [batch, seq_len, hidden_dim * 2]
        
        # Chạy qua Attention để lấy vector ngữ cảnh
        context = self.attention(out) # [batch, hidden_dim * 2]
        return context

In [ ]:
class UserBehaviorModel(nn.Module):
    """
    Kiến trúc tổng thể:
      Input sequence → Encoder → [context + meta] → 6 Classification Heads
    """
    def __init__(self,
                 vocab_size: int,
                 n_classes_list: list,     # [n_class_attr1, ..., n_class_attr6]
                 cfg: dict):
        super().__init__()
        self.cfg = cfg
        model_type = cfg["model_type"]
        E  = cfg["embed_dim"]
        H  = cfg["hidden_dim"]
        L  = cfg["n_layers"]
        Nh = cfg["n_heads"]
        T  = cfg["max_seq_len"]
        D  = cfg["dropout"]

        # ── Encoder
        if model_type == "gru":
            self.encoder = GRUEncoder(vocab_size, E, H, L, D)
        # elif model_type == "cnn":
        #     self.encoder = CNNEncoder(vocab_size, E, H, D)
        # elif model_type == "transformer":
        #     self.encoder = TransformerEncoder(vocab_size, E, H, L, Nh, T, D)
        else:
            raise ValueError(f"model_type không hợp lệ: {model_type}")

        enc_dim  = self.encoder.out_dim
        meta_dim = 2 #3                       # seq_len_norm, unique_ratio, first_token
        in_dim   = enc_dim + meta_dim       # input cho mỗi head

        # ── 6 Classification Heads (độc lập)
        self.heads = nn.ModuleList([
            ClassificationHead(in_dim, cfg["head_hidden"], n_cls, D)
            for n_cls in n_classes_list
        ])

        self._init_weights()

    def _init_weights(self):
        """Khởi tạo trọng số hợp lý để training ổn định hơn."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0, std=0.01)
                if m.padding_idx is not None:
                    nn.init.zeros_(m.weight[m.padding_idx])

    def forward(self, seq, mask, meta):
        """
        seq:  [B, T]  — token indices
        mask: [B, T]  — True = token thực
        meta: [B, 3]  — meta features
        Returns: list of 6 logit tensors, mỗi tensor [B, n_class_i]
        """
        context = self.encoder(seq, mask)           # [B, enc_dim]
        x = torch.cat([context, meta], dim=-1)      # [B, enc_dim + 3]
        return [head(x) for head in self.heads]     # list of [B, n_cls_i]

### Xây dựng hàm loss để tối ưu hóa mô hình

In [11]:
class FocalLoss(nn.Module):
    """
    Focal Loss: FL = -α(1-p)^γ * log(p)
    Tập trung vào hard examples, giảm ảnh hưởng easy examples.
    Dùng cho attr có class imbalance nặng.
    """
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

In [12]:
class MultiOutputLoss(nn.Module):
    """
    Loss tổng hợp cho 6 heads:
    - Mỗi head dùng CE thường hoặc Focal Loss tuỳ imbalance ratio.
    - Có thể gán trọng số khác nhau cho từng head.
    """
    def __init__(self, n_classes_list, imbalance_ratios, device,
                 focal_threshold=5.0, gamma=2.0):
        super().__init__()
        self.loss_fns = nn.ModuleList()
        for i, (n_cls, ir) in enumerate(zip(n_classes_list, imbalance_ratios)):
            print(f"  attr_{i+1}: Cross-Entropy   (IR={ir:.1f})")
            self.loss_fns.append(nn.CrossEntropyLoss())

    def forward(self, logits_list, targets):
        """
        logits_list: list of 6 tensors [B, n_cls_i]
        targets:     [B, 6] LongTensor
        """
        total = 0
        losses = []
        for i, (loss_fn, logits) in enumerate(zip(self.loss_fns, logits_list)):
            l = loss_fn(logits, targets[:, i])
            losses.append(l.item())
            total += l
        return total, losses

### Một số hàm đơn hỗ trợ vào các quá trình training model

In [13]:
def exact_match_accuracy(preds: np.ndarray, labels: np.ndarray) -> float:
    """
    Exact-Match: chỉ tính điểm khi cả 6 attr đều đúng đồng thời.
    preds, labels: [N, 6]
    """
    return (preds == labels).all(axis=1).mean()

def per_attr_accuracy(preds: np.ndarray, labels: np.ndarray) -> dict:
    """Accuracy riêng cho từng attr (để phân tích điểm mạnh/yếu)."""
    return {f"attr_{i+1}_acc": (preds[:, i] == labels[:, i]).mean()
            for i in range(6)}

In [14]:
def load_csv(path):
    return pd.read_csv(path, index_col="id")

def compute_class_info(Y: pd.DataFrame):
    """
    Trả về:
      - n_classes_list: số class mỗi attr
      - imbalance_ratios: IR mỗi attr
      - class_maps: dict mapping raw_label → index (để decode khi predict)
    """
    n_classes_list   = []
    imbalance_ratios = []
    class_maps       = {}

    for col in TARGET_COLS:
        vc   = Y[col].value_counts()
        uniq = sorted(Y[col].unique())
        n_classes_list.append(len(uniq))
        ir = vc.max() / vc.min() if vc.min() > 0 else float("inf")
        imbalance_ratios.append(ir)
        class_maps[col] = {j: v for j, v in enumerate(uniq)}  # index → raw value

    return n_classes_list, imbalance_ratios, class_maps

### Các hàm chạy toàn bộ pipeline training

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device, epoch=None):
    """
    Huấn luyện mô hình và evalue mô hình trên tập train để theo dõi loss và EM accuracy qua từng epoch.
     - model: UserBehaviorModel
     - loader: DataLoader cho tập train
     - optimizer: AdamW
     - loss_fn: MultiOutputLoss
     - device: "cuda" hoặc "cpu"
     - epoch: số epoch hiện tại (để hiển thị trên tqdm)
     Returns: (avg_loss, em_accuracy)
     - avg_loss: loss trung bình trên tập train
     - em_accuracy: exact match accuracy trên tập train (để theo dõi, không phải là metric chính)
     Lưu ý: EM accuracy trên tập train thường sẽ thấp hơn trên tập val do mô hình chưa học đủ, nhưng nếu EM train quá cao ngay từ đầu có thể
    """
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Train]", leave=False)
    for batch in pbar:
        seq, mask, meta, labels = [b.to(device) for b in batch]
        optimizer.zero_grad()

        logits_list = model(seq, mask, meta)
        loss, _ = loss_fn(logits_list, labels)
        loss.backward()

        # Gradient clipping — quan trọng với GRU
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()

        total_loss += loss.item()

        # Thu thập predictions
        preds = torch.stack([l.argmax(dim=-1) for l in logits_list], dim=-1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds  = np.concatenate(all_preds,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    em_acc = exact_match_accuracy(all_preds, all_labels)
    return total_loss / len(loader), em_acc


@torch.no_grad()
def evaluate(model, loader, loss_fn, device, epoch=None):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Val]", leave=False)

    for batch in pbar:
        seq, mask, meta, labels = [b.to(device) for b in batch]
        logits_list = model(seq, mask, meta)
        loss, _ = loss_fn(logits_list, labels)
        total_loss += loss.item()

        preds = torch.stack([l.argmax(dim=-1) for l in logits_list], dim=-1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds  = np.concatenate(all_preds,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    em_acc       = exact_match_accuracy(all_preds, all_labels)
    per_attr_acc = per_attr_accuracy(all_preds, all_labels)
    return total_loss / len(loader), em_acc, per_attr_acc, all_preds


@torch.no_grad()
def predict(model, loader, device):
    """Predict trên tập Test (không có label)."""
    model.eval()
    all_preds = []
    for batch in loader:
        seq, mask, meta = [b.to(device) for b in batch]
        logits_list = model(seq, mask, meta)
        preds = torch.stack([l.argmax(dim=-1) for l in logits_list], dim=-1)
        all_preds.append(preds.cpu().numpy())
    return np.concatenate(all_preds, axis=0)

In [16]:
def run_training(cfg, X_train, Y_train, X_val=None, Y_val=None):
    """
    Huấn luyện model.
    - Nếu có X_val/Y_val: train + validate (giai đoạn 1)
    - Nếu không: chỉ train (giai đoạn 2 — gộp dữ liệu)
    """
    device = cfg["device"]
    print(f"\n🖥️  Device: {device}")
    print(f"🔧  Model:  {cfg['model_type'].upper()}")

    # ── Tính class info từ TRAIN (hoặc train+val nếu gộp)
    print("\n📊 Tính Class Info...")
    n_classes_list, imbalance_ratios, class_maps = compute_class_info(Y_train)

    # ── Build Vocabulary từ TRAIN
    print("\n📖 Xây dựng Vocabulary...")
    vocab = Vocabulary()
    vocab.build(X_train)

    # ── Dataset & DataLoader
    train_ds = BehaviorDataset(X_train, vocab, cfg["max_seq_len"], Y_train, n_classes_list)
    train_dl = DataLoader(train_ds, batch_size=cfg["batch_size"],
                          shuffle=True, num_workers=cfg["num_workers"],
                          pin_memory=(device == "cuda"))

    val_dl = None
    if X_val is not None and Y_val is not None:
        val_ds = BehaviorDataset(X_val, vocab, cfg["max_seq_len"], Y_val, n_classes_list)
        val_dl = DataLoader(val_ds, batch_size=cfg["batch_size"] * 2,
                            shuffle=False, num_workers=cfg["num_workers"])

    # ── Model, Loss, Optimizer
    print("\n🏗️  Khởi tạo Model...")
    model = UserBehaviorModel(vocab.size, n_classes_list, cfg).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"   Tổng params: {total_params:,}")

    print("\n⚖️  Khởi tạo Loss Function...")
    loss_fn = MultiOutputLoss(n_classes_list, imbalance_ratios, device)

    optimizer = AdamW(model.parameters(),
                      lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg["epochs"], eta_min=1e-6)

    # ── Training Loop
    print(f"\n🚀 Bắt đầu Training ({cfg['epochs']} epochs)...\n")
    print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train EM':>9} | "
          f"{'Val Loss':>8} | {'Val EM':>7} | {'LR':>8}")
    print("-" * 65)

    best_val_em  = -1
    patience_cnt = 0
    history      = defaultdict(list)

    for epoch in range(1, cfg["epochs"] + 1):
        train_loss, train_em = train_one_epoch(
            model, train_dl, optimizer, loss_fn, device, epoch
        )
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        history["train_loss"].append(train_loss)
        history["train_em"].append(train_em)

        if val_dl is not None:
            val_loss, val_em, per_attr, _ = evaluate(
                model, val_dl, loss_fn, device, epoch
            )
            history["val_loss"].append(val_loss)
            history["val_em"].append(val_em)

            improved = "✅" if val_em > best_val_em else "  "
            print(f"{epoch:>5} | {train_loss:>10.4f} | {train_em:>8.4f} "
                  f"| {val_loss:>8.4f} | {val_em:>7.4f} | {current_lr:.2e} {improved}")

            if val_em > best_val_em:
                best_val_em  = val_em
                patience_cnt = 0
                # Lưu checkpoint tốt nhất
                torch.save({
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "vocab": vocab,
                    "n_classes_list": n_classes_list,
                    "class_maps": class_maps,
                    "val_em": val_em,
                    "cfg": cfg,
                }, cfg["save_path"])
                print(f"       💾 Đã lưu model tốt nhất (val EM={val_em:.4f})")
            else:
                patience_cnt += 1
                if patience_cnt >= cfg["patience"]:
                    print(f"\n⏹️  Early stopping tại epoch {epoch} "
                          f"(patience={cfg['patience']})")
                    break
        else:
            # Chế độ không có val (giai đoạn 2 — train trên toàn bộ data)
            print(f"{epoch:>5} | {train_loss:>10.4f} | {train_em:>8.4f} "
                  f"|    N/A   |   N/A  | {current_lr:.2e}")

    # Lưu model cuối (giai đoạn 2 không có early stopping)
    if val_dl is None:
        torch.save({
            "epoch": cfg["epochs"],
            "model_state": model.state_dict(),
            "vocab": vocab,
            "n_classes_list": n_classes_list,
            "class_maps": class_maps,
            "val_em": None,
            "cfg": cfg,
        }, cfg["save_path"])
        print(f"\n💾 Đã lưu model cuối: {cfg['save_path']}")
    else:
        print(f"\n🏆 Best Val Exact-Match Accuracy: {best_val_em:.4f}")

        # Load lại model tốt nhất để in per-attr accuracy
        ckpt = torch.load(cfg["save_path"], map_location=device)
        model.load_state_dict(ckpt["model_state"])
        _, _, per_attr, _ = evaluate(model, val_dl, loss_fn, device)
        print("\n📊 Per-Attribute Accuracy (Val):")
        for k, v in per_attr.items():
            print(f"   {k}: {v:.4f}")

    return model, vocab, n_classes_list, class_maps, history

In [17]:
def make_submission(cfg, X_test, model, vocab, n_classes_list, class_maps):
    """
    Predict trên X_test và tạo file submission.csv
    Tự động decode index → raw label → ép kiểu UINT16
    """
    device = cfg["device"]
    model.to(device).eval()

    # Tạo dataset test (không có Y)
    test_ds = BehaviorDataset(X_test, vocab, cfg["max_seq_len"],
                              Y=None, n_classes=None)
    test_dl = DataLoader(test_ds, batch_size=cfg["batch_size"] * 2,
                         shuffle=False, num_workers=cfg["num_workers"])

    preds_idx = predict(model, test_dl, device)   # [N, 6] — index

    # Decode index → raw label
    preds_raw = np.zeros_like(preds_idx, dtype=np.uint16)
    for i, col in enumerate(TARGET_COLS):
        cmap = class_maps[col]
        preds_raw[:, i] = np.array(
            [cmap.get(idx, 0) for idx in preds_idx[:, i]], dtype=np.uint16
        )

    # Tạo DataFrame submission
    sub = pd.DataFrame(preds_raw, columns=TARGET_COLS, index=X_test.index)
    sub.index.name = "id"
    sub = sub.astype(np.uint16)     # Đảm bảo UINT16 như yêu cầu
    sub.to_csv(cfg["output_file"])
    print(f"\n✅ Submission đã lưu: {cfg['output_file']}")
    print(sub.head())
    return sub

### Cuối cùng chạy toàn bộ pipelien training

In [18]:
def main(args):
    print(args)

    # Override config từ args
    CFG.update({
        "model_type":  args.model,
        "epochs":      args.epochs,
        "batch_size":  args.batch_size,
        "lr":          args.lr,
        "max_seq_len": args.max_seq_len,
    })

    print("=" * 65)
    print("  USER BEHAVIOR PREDICTION — DATAFLOW 2026")
    print(f"  Mode: {args.mode.upper()} | Model: {CFG['model_type'].upper()}")
    print("=" * 65)

    # ── Load dữ liệu
    print("\n📂 Load dữ liệu...")
    X_train = load_csv(CFG["x_train"])
    Y_train = load_csv(CFG["y_train"])
    X_val   = load_csv(CFG["x_val"])
    Y_val   = load_csv(CFG["y_val"])

    if args.mode == "train":
        # ── GIAI ĐOẠN 1: Train + Validate riêng biệt
        print(f"\n[Giai đoạn 1] Train={len(X_train)} | Val={len(X_val)}")
        model, vocab, n_classes_list, class_maps, history = run_training(
            CFG, X_train, Y_train, X_val, Y_val
        )

    elif args.mode == "submit":
        # ── GIAI ĐOẠN 2: Gộp Train + Val → Train lại → Predict Test
        print("\n[Giai đoạn 2] Gộp Train + Val để tối đa dữ liệu huấn luyện...")
        X_all = pd.concat([X_train, X_val])
        Y_all = pd.concat([Y_train, Y_val])
        print(f"  Tổng dữ liệu train: {len(X_all)} mẫu")

        # Không có val trong giai đoạn 2 → train toàn bộ epochs
        CFG["patience"] = CFG["epochs"]  # Tắt early stopping
        model, vocab, n_classes_list, class_maps, _ = run_training(
            CFG, X_all, Y_all, X_val=None, Y_val=None
        )

        # Predict test
        print("\n🔮 Predict trên tập Test...")
        X_test = load_csv(CFG["x_test"])
        make_submission(CFG, X_test, model, vocab, n_classes_list, class_maps)

    print("\n✅ HOÀN THÀNH!")

In [19]:
from types import SimpleNamespace

# args = SimpleNamespace(
#     mode="train",
#     model="gru",
#     epochs=CFG["epochs"],
#     batch_size=CFG["batch_size"],
#     lr=CFG["lr"],
#     max_seq_len=CFG["max_seq_len"],
# )

# main(args)

In [ ]:
args = SimpleNamespace(
    mode="submit",
    model="gru",
    epochs=CFG["epochs"],
    batch_size=CFG["batch_size"],
    lr=CFG["lr"],
    max_seq_len=CFG["max_seq_len"],
)

main(args)

namespace(mode='submit', model='gru', epochs=100, batch_size=256, lr=0.001, max_seq_len=30)
  USER BEHAVIOR PREDICTION — DATAFLOW 2026
  Mode: SUBMIT | Model: GRU

📂 Load dữ liệu...

[Giai đoạn 2] Gộp Train + Val để tối đa dữ liệu huấn luyện...
  Tổng dữ liệu train: 58200 mẫu

🖥️  Device: cuda
🔧  Model:  GRU

📊 Tính Class Info...

📖 Xây dựng Vocabulary...
  Vocabulary size: 256 (PAD=0, UNK=1, tokens=254)

🏗️  Khởi tạo Model...
   Tổng params: 8,094,621

⚖️  Khởi tạo Loss Function...
  attr_1: Cross-Entropy   (IR=99.2)
  attr_2: Cross-Entropy   (IR=12.9)
  attr_3: Cross-Entropy   (IR=1.2)
  attr_4: Cross-Entropy   (IR=99.3)
  attr_5: Cross-Entropy   (IR=12.9)
  attr_6: Cross-Entropy   (IR=1.2)

🚀 Bắt đầu Training (100 epochs)...

Epoch | Train Loss |  Train EM | Val Loss |  Val EM |       LR
-----------------------------------------------------------------


Epoch 1 [Train]:   0%|          | 0/228 [00:00<?, ?it/s]

    1 |    13.5162 |   0.0002 |    N/A   |   N/A  | 1.00e-03


Epoch 2 [Train]:   0%|          | 0/228 [00:00<?, ?it/s]

    2 |     4.7457 |   0.4052 |    N/A   |   N/A  | 9.99e-04


...

   98 |     0.0011 |   0.9998 |    N/A   |   N/A  | 1.99e-06


Epoch 99 [Train]:   0%|          | 0/228 [00:00<?, ?it/s]

   99 |     0.0012 |   0.9999 |    N/A   |   N/A  | 1.25e-06


Epoch 100 [Train]:   0%|          | 0/228 [00:00<?, ?it/s]

  100 |     0.0011 |   0.9999 |    N/A   |   N/A  | 1.00e-06

💾 Đã lưu model cuối: /kaggle/working/model_bigru.pt

🔮 Predict trên tập Test...

✅ Submission đã lưu: /kaggle/working/submission.csv
       attr_1  attr_2  attr_3  attr_4  attr_5  attr_6
id                                                   
gpbfd      11       1      84      12       1      96
w22ee       9      19      55       3      12      63
wyw95       1       1      98       3       1      53
izx4w       5       1      72       1       1      77
c6o2d      11       1      22      12       1       1

✅ HOÀN THÀNH!
